In [1]:
import pandas as pd
from io import StringIO

data =  """user_id,signup_days_ago,sessions_last_7d,avg_session_minutes,clicked_marketing,device_type,country,upgraded
1,5,3,8.0,Yes,Mobile,IN,0
2,45,12,15.0,No,Desktop,US,1
3,30,8,10.5,Yes,Mobile,IN,0
4,90,20,25.0,No,Desktop,UK,1
5,10,4,9.0,Yes,Tablet,IN,0
6,60,16,18.0,No,Desktop,US,1
7,15,6,11.0,Yes,Mobile,UK,0
8,120,25,30.0,No,Desktop,US,1
9,3,2,6.0,Yes,Mobile,IN,0
10,75,18,22.0,No,Tablet,UK,1"""

df = pd.read_csv(StringIO(data))

df

,user_id,signup_days_ago,sessions_last_7d,avg_session_minutes,clicked_marketing,device_type,country,upgraded
0,1,5,3,8.0,Yes,Mobile,IN,0
1,2,45,12,15.0,No,Desktop,US,1
2,3,30,8,10.5,Yes,Mobile,IN,0
3,4,90,20,25.0,No,Desktop,UK,1
4,5,10,4,9.0,Yes,Tablet,IN,0
5,6,60,16,18.0,No,Desktop,US,1
6,7,15,6,11.0,Yes,Mobile,UK,0
7,8,120,25,30.0,No,Desktop,US,1
8,9,3,2,6.0,Yes,Mobile,IN,0
9,10,75,18,22.0,No,Tablet,UK,1


In [2]:
df.describe()

,user_id,signup_days_ago,sessions_last_7d,avg_session_minutes,upgraded
count,10.00000,10.000000,10.000000,10.000000,10.000000
mean,5.50000,45.300000,11.400000,15.450000,0.500000
std,3.02765,40.122175,8.016649,8.050017,0.527046
min,1.00000,3.000000,2.000000,6.000000,0.000000
25%,3.25000,11.250000,4.500000,9.375000,0.000000
50%,5.50000,37.500000,10.000000,13.000000,0.500000
75%,7.75000,71.250000,17.500000,21.000000,1.000000
max,10.00000,120.000000,25.000000,30.000000,1.000000


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   user_id              10 non-null     int64  
 1   signup_days_ago      10 non-null     int64  
 2   sessions_last_7d     10 non-null     int64  
 3   avg_session_minutes  10 non-null     float64
 4   clicked_marketing    10 non-null     object 
 5   device_type          10 non-null     object 
 6   country              10 non-null     object 
 7   upgraded             10 non-null     int64  
dtypes: float64(1), int64(4), object(3)
memory usage: 772.0+ bytes


In [5]:
df.isnull().sum()

user_id                0
signup_days_ago        0
sessions_last_7d       0
avg_session_minutes    0
clicked_marketing      0
device_type            0
country                0
upgraded               0
dtype: int64

In [7]:
df.nunique()

user_id                10
signup_days_ago        10
sessions_last_7d       10
avg_session_minutes    10
clicked_marketing       2
device_type             3
country                 3
upgraded                2
dtype: int64

In [30]:
y = df["upgraded"]

features = ["signup_days_ago","sessions_last_7d", "avg_session_minutes", "clicked_marketing", "device_type", "country"]
X = df[features]

In [62]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

categorical_features = ["clicked_marketing","device_type","country"]
numerical_features = ["signup_days_ago","sessions_last_7d","avg_session_minutes"]

preprocessor = ColumnTransformer(
    transformers =[
        ("numerical features", StandardScaler(), numerical_features),
                   ("categorical features", OneHotEncoder(handle_unknown = "ignore"),categorical_features)])

In [63]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split (X,y, random_state = 42, test_size = 0.3)

In [64]:
pipeLR = Pipeline(steps =
                  [("Pre-Processing",preprocessor),
                   ("Classification Logistic",LogisticRegression())])

pipeRF = Pipeline(steps =
                  [("Pre-Processing",preprocessor),
                   ("Classification RandomForest",RandomForestClassifier())])

In [65]:
pipeLR.fit(X_train,y_train)


Pipeline(steps=[('Pre-Processing',
                 ColumnTransformer(transformers=[('numerical features',
                                                  StandardScaler(),
                                                  ['signup_days_ago',
                                                   'sessions_last_7d',
                                                   'avg_session_minutes']),
                                                 ('categorical features',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['clicked_marketing',
                                                   'device_type',
                                                   'country'])])),
                ('Classification Logistic', LogisticRegression())])

In [66]:
pipeRF.fit(X_train,y_train)

Pipeline(steps=[('Pre-Processing',
                 ColumnTransformer(transformers=[('numerical features',
                                                  StandardScaler(),
                                                  ['signup_days_ago',
                                                   'sessions_last_7d',
                                                   'avg_session_minutes']),
                                                 ('categorical features',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['clicked_marketing',
                                                   'device_type',
                                                   'country'])])),
                ('Classification RandomForest', RandomForestClassifier())])

In [74]:
y_predLR = pipeLR.predict(X_test)
y_predLR

array([0, 1, 1], dtype=int64)

In [75]:
y_predRF = pipeRF.predict(X_test)
y_predRF

array([0, 1, 1], dtype=int64)

In [76]:
print(" The score of Logistic Regression {}%".format(round(pipeLR.score(X_test,y_test)*100),0))
print(" The score of Random Forest {}%".format(round(pipeRF.score(X_test,y_test)*100),0))

 The Accuracy score of Logistic Regression 100%
 The Accuracy score of Random Forest 100%


In [77]:
y_test

8    0
1    1
5    1
Name: upgraded, dtype: int64

In [84]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracyLR = accuracy_score(y_test, y_predLR)
accuracyRF = accuracy_score(y_test, y_predRF)

print(accuracyLR)
print(accuracyRF)

classificationLR = classification_report(y_test,y_predLR)
classificationRF = classification_report(y_test,y_predRF)

print(classificationLR)
print(classificationRF)

1.0
1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         2

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         2

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



In [83]:
confusion_matrixLR = confusion_matrix(y_test,y_predLR)
confusion_matrixRF = confusion_matrix(y_test,y_predRF)

print(confusion_matrixLR)
print(confusion_matrixRF)

[[1 0]
 [0 2]]
[[1 0]
 [0 2]]


In [96]:
coefLR = pipeLR.named_steps["Classification Logistic"].coef_[0]
LR_features = pipeLR.named_steps["Pre-Processing"].get_feature_names_out()
LR_importance = pd.DataFrame({"Features":LR_features, "Co-efficients" : coefLR})
print(LR_importance.sort_values("Co-efficients", ascending = False))

                                       Features  Co-efficients
2       numerical features__avg_session_minutes       0.585856
1          numerical features__sessions_last_7d       0.585369
0           numerical features__signup_days_ago       0.564686
3    categorical features__clicked_marketing_No       0.345245
9              categorical features__country_UK       0.185507
5     categorical features__device_type_Desktop       0.157239
7      categorical features__device_type_Tablet       0.108281
10             categorical features__country_US       0.045571
8              categorical features__country_IN      -0.230925
6      categorical features__device_type_Mobile      -0.265367
4   categorical features__clicked_marketing_Yes      -0.345092


In [104]:
RF_importances = pipeRF.named_steps['Classification RandomForest'].feature_importances_
RF_features = pipeRF.named_steps['Pre-Processing'].get_feature_names_out()
RF_importance = pd.DataFrame({"Features" : RF_features, "Importances" : RF_importances})

print(RF_importance.sort_values("Importances", ascending = False))

                                       Features  Importances
3    categorical features__clicked_marketing_No     0.214410
0           numerical features__signup_days_ago     0.205668
4   categorical features__clicked_marketing_Yes     0.186806
2       numerical features__avg_session_minutes     0.119141
1          numerical features__sessions_last_7d     0.107361
5     categorical features__device_type_Desktop     0.068056
8              categorical features__country_IN     0.064861
6      categorical features__device_type_Mobile     0.017943
7      categorical features__device_type_Tablet     0.011415
10             categorical features__country_US     0.004340
9              categorical features__country_UK     0.000000


In [110]:
import pandas as pd
from sklearn.inspection import permutation_importance
result = permutation_importance(pipeRF,X_test,y_test,n_repeats = 10, random_state = 42)
importances = pd.DataFrame({"Features" : X_test.columns, "Importance" : result.importances_mean})
print(importances.sort_values("Importance", ascending = False))

              Features  Importance
3    clicked_marketing         0.2
4          device_type         0.1
0      signup_days_ago         0.0
1     sessions_last_7d         0.0
2  avg_session_minutes         0.0
5              country         0.0
